<a href="https://colab.research.google.com/github/Ammareli/Ammareli/blob/main/FaceRec_MobileNet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from keras.applications import MobileNet

In [2]:
# MobileNet was designed to work on 224 x 224 pixel input images sizes
img_rows, img_cols = 224, 224

# Re-loads the MobileNet model without the top or FC layers
MobileNet = MobileNet(weights = 'imagenet',
                 include_top = False,
                 input_shape = (img_rows, img_cols, 3))

In [3]:
# Here we freeze the last 4 layers
# Layers are set to trainable as True by default so we make them untrainable now
for layer in MobileNet.layers:
    layer.trainable = False

In [4]:
# Let's print our layers
for (i,layer) in enumerate(MobileNet.layers):
    print(str(i) + " "+ layer.__class__.__name__, layer.trainable)

0 InputLayer False
1 Conv2D False
2 BatchNormalization False
3 ReLU False
4 DepthwiseConv2D False
5 BatchNormalization False
6 ReLU False
7 Conv2D False
8 BatchNormalization False
9 ReLU False
10 ZeroPadding2D False
11 DepthwiseConv2D False
12 BatchNormalization False
13 ReLU False
14 Conv2D False
15 BatchNormalization False
16 ReLU False
17 DepthwiseConv2D False
18 BatchNormalization False
19 ReLU False
20 Conv2D False
21 BatchNormalization False
22 ReLU False
23 ZeroPadding2D False
24 DepthwiseConv2D False
25 BatchNormalization False
26 ReLU False
27 Conv2D False
28 BatchNormalization False
29 ReLU False
30 DepthwiseConv2D False
31 BatchNormalization False
32 ReLU False
33 Conv2D False
34 BatchNormalization False
35 ReLU False
36 ZeroPadding2D False
37 DepthwiseConv2D False
38 BatchNormalization False
39 ReLU False
40 Conv2D False
41 BatchNormalization False
42 ReLU False
43 DepthwiseConv2D False
44 BatchNormalization False
45 ReLU False
46 Conv2D False
47 BatchNormalization False
48

In [5]:
def head_model(bottom_model, num_classes):
    """creates the top or head of the model that will be
    placed ontop of the bottom layers"""

    top_model = bottom_model.output
    top_model = GlobalAveragePooling2D()(top_model)
    top_model = Dense(1024,activation='relu')(top_model)
    top_model = Dense(1024,activation='relu')(top_model)
    top_model = Dropout(0.25)(top_model)
    top_model = Dense(512,activation='relu')(top_model)
    top_model = Dense(num_classes,activation='softmax')(top_model)
    return top_model


In [6]:
from keras.models import Sequential
from keras.layers import Dense, Dropout, Activation, Flatten, GlobalAveragePooling2D
from keras.layers import Conv2D, MaxPooling2D, ZeroPadding2D
from keras.layers import BatchNormalization
from keras.models import Model

In [7]:
num_classes = 9

FC_Head = head_model(MobileNet, num_classes)

#Now attaching the top and bottom layers together to form a new model
model = Model(inputs = MobileNet.input, outputs = FC_Head)

print(model.summary())

Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 224, 224, 3)]     0         
                                                                 
 conv1 (Conv2D)              (None, 112, 112, 32)      864       
                                                                 
 conv1_bn (BatchNormalizati  (None, 112, 112, 32)      128       
 on)                                                             
                                                                 
 conv1_relu (ReLU)           (None, 112, 112, 32)      0         
                                                                 
 conv_dw_1 (DepthwiseConv2D  (None, 112, 112, 32)      288       
 )                                                               
                                                                 
 conv_dw_1_bn (BatchNormali  (None, 112, 112, 32)      128   

In [8]:
path = "/content/drive/MyDrive/Data Sets/data"

In [9]:
import os
os.listdir(path)

['Aftab',
 'Ammar',
 'Atiqa',
 'Gulam',
 'Ibrahim',
 'makdoom',
 'Maria',
 'Mohsin',
 'shahzad']

In [10]:
from keras.preprocessing.image import ImageDataGenerator




In [11]:
# Let's use some data augmentaiton
train_datagen = ImageDataGenerator(
      rescale=1./255,
      rotation_range=45,
      width_shift_range=0.3,
      height_shift_range=0.3,
      horizontal_flip=True,
      fill_mode='nearest',
      validation_split= 0.2
      )

In [12]:
train_generator = train_datagen.flow_from_directory(
    path,
    target_size=(150, 150),  # Adjust the target size as needed
    batch_size=32,
    class_mode='categorical',
    subset='training',

)

Found 3560 images belonging to 9 classes.


In [13]:
validation_generator  = train_datagen.flow_from_directory(
    path,
    target_size=(150, 150),  # Adjust the target size as needed
    batch_size=32,
    class_mode='categorical',  # Set to 'categorical' for multi-class classification
    subset='validation', )

Found 886 images belonging to 9 classes.


In [14]:
from keras.optimizers import RMSprop
from keras.callbacks import ModelCheckpoint, EarlyStopping


checkpoint = ModelCheckpoint("faces.h5",
                             monitor="val_loss",
                             mode="min",
                             save_best_only = True,
                             verbose=1)

earlystop = EarlyStopping(monitor = 'val_loss',
                          min_delta = 0,
                          patience = 3,
                          verbose = 1,
                          restore_best_weights = True)

# we put our call backs into a callback list
callbacks = [earlystop, checkpoint]

# We use a very small learning rate
#for multi classification categorical_crossentropy is used
model.compile(loss = 'categorical_crossentropy',
              optimizer = RMSprop(lr = 0.001),
              metrics = ['accuracy'])

In [16]:
# Enter the number of training and validation samples here
nb_train_samples = 3560
nb_validation_samples = 886

# We only train 5 EPOCHS
epochs = 2
batch_size = 32

history = model.fit_generator(
    train_generator,
    steps_per_epoch = nb_train_samples // batch_size,
    epochs = epochs,
    callbacks = callbacks,
    validation_data = validation_generator,
    validation_steps = nb_validation_samples // batch_size)

<ipython-input-16-f0e4fe8ad935>:9: UserWarning: `Model.fit_generator` is deprecated and will be removed in a future version. Please use `Model.fit`, which supports generators.
  history = model.fit_generator(


Epoch 1/2
111/111 [==============================] - ETA: 0s - loss: 0.0930 - accuracy: 0.9782
Epoch 1: val_loss improved from 0.01779 to 0.01690, saving model to faces.h5
111/111 [==============================] - 45s 403ms/step - loss: 0.0930 - accuracy: 0.9782 - val_loss: 0.0169 - val_accuracy: 0.9931
Epoch 2/2
111/111 [==============================] - ETA: 0s - loss: 0.0693 - accuracy: 0.9836
Epoch 2: val_loss improved from 0.01690 to 0.00856, saving model to faces.h5
111/111 [==============================] - 44s 393ms/step - loss: 0.0693 - accuracy: 0.9836 - val_loss: 0.0086 - val_accuracy: 0.9977


In [17]:
model.save("face_classifier.h5")

/usr/local/lib/python3.10/dist-packages/keras/src/engine/training.py:3079: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(
